# Cost Curves You Can Feel

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 6 §6.1–6.3 · type: drill*

**The promise:** by the end you can pick the right structure by recognizing its cost curve, and catch the accidental-quadratic patterns that pass every unit test and melt the first week real data arrives.

This is a drill: short tasks, quick checks, minimal narration. It runs fully offline and free — nothing here calls a model or the network. The only "measurement" is `time.perf_counter`.

## 🧠 Why this matters

You do not need the formal definition of Big-O. You need one reflex: *when the input grows 10×, what happens to the cost?* O(1) shrugs; O(log n) barely flinches; O(n) costs 10×; O(n log n) a bit worse; O(n²) costs **100×** — fine at a hundred items, fatal at a million.

The structure you choose *is* the algorithm you get. `x in some_list` scans every element; `x in some_set` hashes straight to the answer. Same line of code, one changed type, and the difference between a one-second job and an overnight one. The half-dozen daily-driver structures — `dict`/`set`, `list`, `deque`, heap, tree/`bisect`, graph — are 90% of the algorithmic skill a production AI system demands. See §6.1–§6.2.

## Objectives & prereqs

**By the end you can:**
- Predict a cost curve in *feel* (O(1) / log n / n / n log n / n²) before you write the loop.
- Spot the quadratic-suspect patterns by eye: `in` on a list, `pop(0)`/`remove` in a loop, `+=` on strings, nested list scans.
- Reach for the right one of the six daily-driver structures by its killer operation.
- Implement the book's `chunk(text, size, overlap)` and tolerant `extract_json` — and budget in *tokens, not characters*.

**Prereqs:** Chapter 4 (Python fluency). No prior notebook required.

**Packages:** the standard library only (`time`, `collections`, `heapq`, `bisect`, `json`, `re`). Nothing to install beyond `requirements.txt`.

In [ ]:
# --- Setup -------------------------------------------------------------------
import os
import random
import time

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default) keeps everything offline, free, and deterministic. This drill
# makes no model calls at all, so MOCK only governs the data SIZE: small in mock
# (fast CI), larger when you opt in to feel the quadratic curve bite for real.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(6)  # determinism for the generated id lists below

# The quadratic demo is O(n^2): keep n modest in mock so the notebook stays snappy.
BIG_N = 4_000 if MOCK else 40_000

print(f"MOCK mode: {MOCK}  (offline, free)")
print(f"quadratic-demo size BIG_N = {BIG_N:,}"
      f"  (set COMPANION_MOCK=0 to feel a bigger melt)")

## 1 · The only Big-O question that matters

Input grows 10× — what happens to cost? Here is the whole table in *feel*, with the cost at n=10 normalized to 1. Read the n=1,000,000 column: that gap is why the structure you reach for decides whether a feature ships or pages you at 3am.

In [ ]:
import math


def cost(curve: str, n: int) -> float:
    return {
        "O(1)":       1.0,
        "O(log n)":   math.log2(n),
        "O(n)":       n,
        "O(n log n)": n * math.log2(n),
        "O(n^2)":     n * n,
    }[curve]


curves = ["O(1)", "O(log n)", "O(n)", "O(n log n)", "O(n^2)"]
sizes = [10, 100, 1_000, 1_000_000]

header = f"{'curve':<11}" + "".join(f"{('n=' + format(n, ',')):>16}" for n in sizes)
print(header)
print("-" * len(header))
for c in curves:
    base = cost(c, 10)  # normalize so n=10 reads as 1x
    row = f"{c:<11}" + "".join(f"{(cost(c, n) / base):>15,.0f}x" for n in sizes)
    print(row)

**What you just saw.** O(1) and O(log n) stay flat enough to ignore. O(n) and O(n log n) grow with the data but stay sane. O(n²) explodes — a million-item job is ten *billion* times the ten-item one. You will not eyeball the exact number; you only need to recognize *which curve you are on*.

## 2 · 🔮 Predict: the dedup loop that melts

Below are the chapter's two dedup loops, the only difference being one type. The first tracks seen ids in a **list** (`id in list` is an O(n) scan, done n times → O(n²)); the second uses a **set** (`id in set` hashes, O(1) average → O(n) overall).

**Predict before running:** at `BIG_N` items, what is the runtime *ratio* between the list version and the set version? 2×? 10×? 100× or more? Write your guess down, then run the next cell.

In [ ]:
# A list of ids with ~50% duplicates, so dedup actually has work to do.
ids = [f"doc-{random.randrange(BIG_N // 2)}" for _ in range(BIG_N)]


def dedup_list(items):
    seen = []                      # quadratic: list lookup inside a loop
    out = []
    for x in items:
        if x in seen:              # O(n) scan, n times -> O(n^2)
            continue
        seen.append(x)
        out.append(x)
    return out


def dedup_set(items):
    seen = set()                   # linear: hash lookup inside a loop
    out = []
    for x in items:
        if x in seen:              # O(1) average
            continue
        seen.add(x)
        out.append(x)
    return out


def timed(fn, items):
    start = time.perf_counter()
    result = fn(items)
    return result, time.perf_counter() - start


# Warm up at a tiny scale first: both look instant, which is exactly the trap.
small = ids[:100]
_, t_list_small = timed(dedup_list, small)
_, t_set_small = timed(dedup_set, small)
print(f"n=100      list {t_list_small * 1e6:8.1f} us   set {t_set_small * 1e6:8.1f} us"
      f"   (both 'instant' -- the test scale that hides the bug)")

out_list, t_list = timed(dedup_list, ids)
out_set, t_set = timed(dedup_set, ids)
assert out_list == out_set, "both must dedup identically (order-preserving)"

ratio = t_list / t_set if t_set else float("inf")
print(f"n={BIG_N:<8,} list {t_list * 1e3:8.1f} ms   set {t_set * 1e3:8.1f} ms"
      f"   -> list is ~{ratio:,.0f}x slower")

**What you just saw.** At 100 items both are microseconds — every unit test passes. At `BIG_N` the list version is *dramatically* slower, and the gap **widens** as n grows (it is the O(n²) vs O(n) curve from section 1). In a RAG ingestion pipeline deduping a million chunks, this is the difference between a one-second pass and an overnight one. Same code shape; one changed type.

## 3 · The quadratic-suspect gallery

Quadratic behavior almost never announces itself. Train your eye on the four most common disguises — each passes at test scale and melts in production. For each, the fix is a structure swap, not a clever algorithm.

In [ ]:
from collections import deque

n = 2_000 if MOCK else 20_000

# (a) Building a string with += in a loop. Strings are IMMUTABLE: each += copies
#     everything so far -> O(n^2). Fix: collect parts, "".join(parts) once -> O(n).
start = time.perf_counter()
s = ""
for i in range(n):
    s += "x"                       # copies the whole string every step
t_concat = time.perf_counter() - start

start = time.perf_counter()
parts = []
for i in range(n):
    parts.append("x")
s2 = "".join(parts)                # one allocation
t_join = time.perf_counter() - start
assert s == s2

# (b) Using a list as a FIFO queue: list.pop(0) shifts every element -> O(n) per
#     pop -> O(n^2) to drain. Fix: collections.deque, O(1) at both ends.
start = time.perf_counter()
q = list(range(n))
while q:
    q.pop(0)                       # shifts all remaining elements left
t_listq = time.perf_counter() - start

start = time.perf_counter()
dq = deque(range(n))
while dq:
    dq.popleft()                   # O(1)
t_deque = time.perf_counter() - start

print(f"(a) string +=  {t_concat * 1e3:7.2f} ms   vs  ''.join  {t_join * 1e3:7.2f} ms")
print(f"(b) list.pop(0){t_listq * 1e3:7.2f} ms   vs  deque    {t_deque * 1e3:7.2f} ms")

In [ ]:
# (c) "Match up two lists" with a nested scan -> O(n*m). Fix: build a dict in ONE
#     pass, then look up in O(1). This is the single most common quadratic in data
#     glue code -- and collections.Counter / defaultdict package the same idiom.
users = [{"id": i, "name": f"user{i}"} for i in range(n)]
orders = [{"user_id": i, "total": i * 2} for i in range(n)]

start = time.perf_counter()
joined_slow = []
for o in orders:
    for u in users:                # inner scan over ALL users, per order -> O(n*m)
        if u["id"] == o["user_id"]:
            joined_slow.append((u["name"], o["total"]))
            break
t_nested = time.perf_counter() - start

start = time.perf_counter()
by_id = {u["id"]: u for u in users}     # build the index once
joined_fast = [(by_id[o["user_id"]]["name"], o["total"]) for o in orders]  # O(1) lookups
t_indexed = time.perf_counter() - start
assert joined_slow == joined_fast

ratio = t_nested / t_indexed if t_indexed else float("inf")
print(f"(c) nested scan {t_nested * 1e3:8.2f} ms   vs  dict index {t_indexed * 1e3:8.2f} ms"
      f"   -> ~{ratio:,.0f}x")

**What you just saw.** Every "slow" column shares a tell: a loop body that touches a *growing* collection — `+=` recopying, `pop(0)` shifting, an inner scan over the whole other list. The reflex: **any loop whose body scans a collection that grows deserves three seconds of suspicion**, and the fix is almost always "reach for a different structure."

## 4 · The six daily drivers, each by its killer op

This is the book's reach-for-it table, made runnable. You will pick correctly almost every time by asking one question: *what is the operation I do most, and which structure makes it cheap?*

In [ ]:
import bisect
import heapq
from collections import Counter, defaultdict, deque

# dict / set -- O(1) lookup: matching, counting, dedup, caching by key.
counts = Counter("the quick brown fox the lazy dog the".split())
print("dict/set  O(1) lookup  ->", counts.most_common(2))

# list -- O(1) append/index: ordered, append-mostly data, small scans.
log = []
log.append("event-a")
log.append("event-b")
print("list      O(1) append  ->", log[-1])

# deque -- O(1) both ends: FIFO queues, sliding windows, BFS frontiers.
recent = deque(maxlen=3)           # a bounded ring buffer for the last 3 events
for e in ["a", "b", "c", "d"]:
    recent.append(e)
print("deque     O(1) ends    ->", list(recent), "(oldest auto-evicted)")

# heap -- O(log n) min/max: top-k, priority scheduling, timers.
scored = [(0.21, "d1"), (0.95, "d2"), (0.40, "d3"), (0.88, "d4"), (0.11, "d5")]
print("heap      O(log n)     ->", heapq.nlargest(2, scored), "(top-2, no full sort)")

# tree / bisect -- O(log n) ordered ops: ranges, sorted order, DB indexes.
timeline = [10, 20, 30, 40, 50]   # sorted timestamps
lo = bisect.bisect_left(timeline, 25)
hi = bisect.bisect_right(timeline, 45)
print("bisect    O(log n)     ->", timeline[lo:hi], "(range query 25..45)")

# graph -- traversal: workflows, plans, dependencies, HNSW. A dict of edges.
graph = {"research": {"draft"}, "draft": {"review"}, "review": set()}
print("graph     traversal    ->", graph["research"], "(neighbors of 'research')")

In [ ]:
# A tiny breadth-first traversal over that dict-of-edges -- one of the two graph
# algorithms worth owning (the other, topological sort, is the next notebook's build).
def bfs(graph, start):
    seen, order, frontier = {start}, [], deque([start])
    while frontier:
        node = frontier.popleft()  # deque as the BFS frontier -> O(1)
        order.append(node)
        for nxt in graph.get(node, ()):
            if nxt not in seen:    # set membership -> O(1) -- no revisits
                seen.add(nxt)
                frontier.append(nxt)
    return order


print("BFS order from 'research':", bfs(graph, "research"))
# Note the structure stack: deque (frontier) + set (visited) + dict (edges).
# Three daily drivers cooperating -- that is what real algorithmic code looks like.

**Reach-for-it summary.** `dict`/`set` for lookup, matching, dedup, caching. `list` for ordered append-mostly data. `deque` for queues and sliding windows. heap for top-k and priorities. `bisect`/tree for ordered and range queries. graph (dict of edges) for workflows and plans. Choosing among these six well *is* the algorithmic skill the job demands.

## 5 · 🔧 Build the sliding-window `chunk` — mind the off-by-one

Chunking — splitting a document into overlapping windows for embedding and retrieval (Chapter 13) — is the recurring string problem. The window-and-overlap skeleton is three lines, and the off-by-one space is exactly where the bugs live. This is the chapter's `chunk`, verbatim.

In [ ]:
def chunk(text: str, size: int = 1000, overlap: int = 200) -> list[str]:
    """Fixed-size windows; consecutive chunks share `overlap` characters."""
    if size <= overlap:
        raise ValueError("size must exceed overlap")
    step = size - overlap
    return [text[i : i + size] for i in range(0, max(len(text) - overlap, 1), step)]


# Probe the off-by-one space with a tiny, countable input.
blob = "".join(str(i % 10) for i in range(25))  # "0123456789012..." length 25
chunks = chunk(blob, size=10, overlap=3)
for i, c in enumerate(chunks):
    print(f"chunk {i}: {c!r}  (len {len(c)})")

# Adjacent chunks must SHARE exactly `overlap` characters -- verify the seam.
overlap = 3
for a, b in zip(chunks, chunks[1:]):
    assert a[-overlap:] == b[:overlap], "overlap seam broken!"
print("\nseam check passed: every adjacent pair shares the last/first 3 chars")

In [ ]:
# ⚠️ size <= overlap would never advance (step <= 0) -> an infinite/garbage split.
# The guard turns that silent hang into a loud, early error. Confirm it fires:
try:
    chunk("whatever", size=5, overlap=5)
except ValueError as exc:
    print("correctly rejected size == overlap:", exc)

# Budget in TOKENS, not characters. The model sees tokens (Chapter 8), and English
# runs ~3-4 chars/token. A 1000-CHARACTER chunk is only ~250-330 tokens -- size your
# windows against the model's token limit, not the character length you can see.
CHARS_PER_TOKEN = 4  # rough English rule of thumb; count exactly with the tokenizer
doc_chars = 8000
print(f"\n{doc_chars} chars  ~=  {doc_chars // CHARS_PER_TOKEN} tokens (rough)")
print("Rule: budget windows + overlap in tokens; this char count is only a proxy.")

**What you just saw.** Each chunk is `size` chars (the last may be shorter), adjacent chunks share exactly `overlap`, and `size <= overlap` is rejected before it can hang. Production chunkers split on semantic boundaries (paragraphs, sentences, code blocks), but this window-and-overlap skeleton stays the same — and you *always* budget in tokens.

## 6 · Tolerant `extract_json` from messy model output

Parsing model output deserves paranoia: you will constantly pull structure from text that is *almost* what you asked for — JSON wrapped in markdown fences, a chatty preamble before the payload. The chapter's defensive extractor finds the JSON-ish span with a regex, then hands it to a real parser.

In [ ]:
import json
import re


def extract_json(text: str) -> dict:
    """Pull the first JSON object out of model output, fences and all."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match is None:
        raise ValueError("no JSON object found in model output")
    return json.loads(match.group())


# A realistic, messy model reply: prose, a fenced code block, then the payload.
messy = (
    "Sure! Here is the result you asked for:\n"
    "```json\n"
    '{"action": "search", "query": "vector databases", "top_k": 5}\n'
    "```\n"
    "Let me know if you'd like anything adjusted."
)

parsed = extract_json(messy)
print("parsed object:", parsed)
print("top_k as int :", parsed["top_k"])

## ⚠️ Pitfall: a regex is for tokens, not for nested structure

`re.search(r"\{.*\}", ...)` greedily grabs from the first `{` to the *last* `}`. On a flat object that is exactly right. But the moment the output contains a *second* object — or nested braces it doesn't understand — the regex matches a span that is no longer valid JSON, and `json.loads` raises. Regex finds the *span*; only a real parser understands the *structure*.

In [ ]:
# Two JSON objects in one reply. The greedy {.*} spans from the first { to the
# last } -- swallowing the gap between them -> not valid JSON.
two_objects = 'First: {"a": 1} and second: {"b": 2}'
try:
    extract_json(two_objects)
except json.JSONDecodeError as exc:
    print("regex grabbed a non-JSON span -> json.loads failed:")
    print("  ", exc)

print(
    "\nThe lesson: regex is the right tool to LOCATE a JSON-ish span or validate an\n"
    "ID format. It is the WRONG tool to PARSE nested structure (real JSON, HTML,\n"
    "code). When a regex grows past one line or sprouts nested groups, switch to a\n"
    "parser -- Chapter 15's structured-output APIs make this rigorous."
)

## 🎯 Senior lens: the costly unit is a *model call*

Every timing in this notebook is in microseconds and milliseconds — a dict lookup against a list scan. In an agent system the expensive unit is six orders of magnitude bigger: a **model call**, billed in dollars and seconds, not nanoseconds. So a senior applies Big-O *where it actually bites* — count cost in model calls and tokens.

The classic trap is the same O(n²) shape from section 2, hiding in plain sight: an agent loop that **re-summarizes the whole conversation history every turn** is O(n²) in tokens over the conversation — turn 50 reprocesses everything from turns 1–49. The data structures here are how you spend cheap operations (a dict, a cache, a `deque` window) to avoid expensive ones (a redundant model call). That trade is what makes the economics of an agent platform work.

## Recap

- Big-O is one reflex: input grows 10×, what happens to cost? O(n²) is fine at 100 items and fatal at a million.
- **Lookups and dedup go through `dict`/`set`**; "match these two lists" means "build a dict first."
- Quadratic suspects: `in` on a list, `pop(0)`/`remove` in a loop, `+=` on strings, nested list scans — each fixed by a structure swap (`set`, `deque`, `"".join`, a dict index).
- The six daily drivers, by killer op: `dict`/`set` (lookup), `list` (append), `deque` (both ends), heap (top-k), `bisect`/tree (ranges), graph (traversal).
- `chunk` is a sliding window with overlap — mind the off-by-one and **budget in tokens, not characters**.
- A regex *locates* a JSON span; a parser *understands* structure — don't ask a regex to do the parser's job.
- The costly unit in agents is a **model call**; apply Big-O to calls and tokens, not just CPU ops.

## Exercises

Each exercise *changes* something and asks you to predict the result before you run. (Solutions arrive in `solutions/` in Phase 2.)

1. **Scale the melt.** Re-run the dedup race at three sizes (`500`, `2000`, `8000`). Predict how the list-vs-set *ratio* changes as n grows — should it stay constant, or widen? Plot or print the ratios and explain which curve you're seeing.
2. **Counter vs hand-rolled.** Rewrite the section-3 nested-scan join using `collections.defaultdict(list)` so one `user_id` can have many orders. Predict whether it stays O(n) and confirm with a timing.
3. **Top-k by hand.** Without `heapq.nlargest`, keep a running top-3 of streaming `(score, id)` pairs using a small heap (`heapq.heappush` + `heapq.heappushpop`). Predict its complexity vs sorting all n, then verify it matches `nlargest`.
4. **Break the seam.** Call `chunk(text, size=10, overlap=10)` and predict what happens *before* running. Then fix `extract_json` to tolerate the two-object input from the pitfall (hint: a non-greedy or brace-counting scan) and predict which object it should return.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## Next

- **Next notebook:** [`06-02-plan-executor-topo-sort.ipynb`](06-02-plan-executor-topo-sort.ipynb) — the chapter's 🔧 Build. You'll turn a dependency dict into a concurrent, cycle-checked task executor that pays the graph's *depth* in latency, not its size.
- **Book:** revisit §6.1–§6.3 (Big-O intuition, the daily structures, strings/parsing) and look ahead to §6.4 (search, ranking, scheduling).
- **Where this leads:** the `chunk` skeleton reappears in the RAG pipeline ([`../../blueprints/rag-pipeline/`](../../blueprints/rag-pipeline/)) and the capstone's `rag/` (Chapter 13); the dedup/dict reflexes underpin every ingestion path you'll write.